# 18.10 Data valuation and Shapley for data

Data Shapley asks how much each training example contributes when credit is averaged fairly across contexts. The value depends on the utility, validation set, model family, and stable data snapshot. Save a copy to Drive to edit.

In [ ]:

import hashlib
import itertools
import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

np.random.seed(18)


def dataquality_ladder():
    """Breast Cancer ladder with progressively degraded data quality."""
    bc = load_breast_cancer()
    X0 = bc.data.astype(float)
    y0 = bc.target
    rng = np.random.default_rng(18)
    rungs = []

    rungs.append(("D1 clean", X0.copy(), y0.copy()))

    y2 = y0.copy()
    flip = rng.random(y2.shape) < 0.15
    y2[flip] = 1 - y2[flip]
    rungs.append(("D2 15% label noise", X0.copy(), y2))

    keep_pos = np.where(y0 == 1)[0]
    keep_neg = np.where(y0 == 0)[0][:30]
    idx = np.concatenate([keep_pos, keep_neg])
    rungs.append(("D3 class imbalance", X0[idx].copy(), y0[idx].copy()))

    X4 = X0 + rng.normal(0.0, X0.std(axis=0) * 0.5, size=X0.shape)
    rungs.append(("D4 heavy feature noise", X4, y0.copy()))

    X5 = X0.copy()
    holes = rng.random(X5.shape) < 0.2
    X5[holes] = np.nan
    col_means = np.nanmean(X5, axis=0)
    X5 = np.where(np.isnan(X5), col_means, X5)
    rungs.append(("D5 20% missing (mean-imputed)", X5, y0.copy()))

    return rungs


def stable_split(X, y):
    return train_test_split(
        X,
        y,
        test_size=0.35,
        random_state=18,
        stratify=y,
    )


def fit_logistic_accuracy(x_train, y_train, x_test, y_test):
    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train)
    x_test_scaled = scaler.transform(x_test)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_train_scaled, y_train)
    pred = clf.predict(x_test_scaled)
    return float(accuracy_score(y_test, pred))


def preview_ladder(rungs):
    for name, X, y in rungs:
        counts = np.bincount(y.astype(int))
        sample = np.round(X[:2, :4], 3)
        print(name)
        print("  shape", X.shape, "class_counts", counts.tolist())
        print("  first_two_by_four")
        print(sample)


## The concept, built once (D1)

For training set $N$, $\phi_i=\sum_{S\subseteq N\setminus\{i\}}\frac{|S|!(n-|S|-1)!}{n!}(U(S\cup\{i\})-U(S))$. We compute exact small-$n$ values for a one-nearest-neighbor utility, then use row values to keep or drop examples.

In [ ]:

def utility_1nn(indices, X_train, y_train, X_val, y_val):
    if len(indices) == 0:
        return 0.0
    idx = np.array(list(indices), dtype=int)
    rows = X_train[idx]
    labels = y_train[idx]
    distances = ((X_val[:, None, :] - rows[None, :, :]) ** 2).sum(axis=2)
    nearest = np.argmin(distances, axis=1)
    pred = labels[nearest]
    return float(accuracy_score(y_val, pred))


def exact_shapley(X_train, y_train, X_val, y_val):
    n = len(y_train)
    values = np.zeros(n)
    all_indices = tuple(range(n))
    utilities = {}

    for r in range(n + 1):
        for subset in itertools.combinations(all_indices, r):
            utilities[subset] = utility_1nn(subset, X_train, y_train, X_val, y_val)

    for i in all_indices:
        others = [j for j in all_indices if j != i]
        for r in range(n):
            weight = math.factorial(r) * math.factorial(n - r - 1) / math.factorial(n)
            for subset in itertools.combinations(others, r):
                subset = tuple(sorted(subset))
                with_i = tuple(sorted(subset + (i,)))
                values[i] += weight * (utilities[with_i] - utilities[subset])

    return values, utilities


X_small = np.array([[-1.0, 0.0], [-1.2, 0.0], [1.0, 0.0]])
y_small = np.array([0, 0, 1])
X_val = np.array([[-1.0, 0.0], [1.0, 0.0]])
y_val = np.array([0, 1])
phi, utilities = exact_shapley(X_small, y_small, X_val, y_val)
print("singletons", utilities[(0,)], utilities[(1,)], utilities[(2,)])
print("phi", np.round(phi, 3), "sum", round(float(phi.sum()), 3))
assert utilities[()] == 0.0
assert utilities[(0,)] == utilities[(1,)] == utilities[(2,)] == 0.5
assert utilities[(0, 2)] == utilities[(1, 2)] == utilities[(0, 1, 2)] == 1.0
assert np.allclose(np.round(phi, 3), [0.250, 0.250, 0.500])
assert round(float(phi.sum()), 3) == 1.000

X_bad = np.array([[-1.0, 0.0], [0.95, 0.0], [1.0, 0.0]])
y_bad = np.array([0, 0, 1])
X_bad_val = np.array([[-1.0, 0.0], [0.96, 0.0]])
y_bad_val = np.array([0, 1])
before = utility_1nn((0, 2), X_bad, y_bad, X_bad_val, y_bad_val)
after = utility_1nn((0, 1, 2), X_bad, y_bad, X_bad_val, y_bad_val)
negative_marginal = after - before
print("negative marginal", negative_marginal)
assert round(negative_marginal, 3) == -0.500


Exact Data Shapley is feasible only for small subsets. This method computes exact values for a stratified sample of rows and drops sampled rows with negative value before training the same classifier.

In [ ]:

def pick_small_stratified(y, max_rows=8):
    rng = np.random.default_rng(1810)
    chosen = []
    classes = np.unique(y)
    per_class = max(1, max_rows // len(classes))
    for cls in classes:
        idx = np.where(y == cls)[0]
        take = min(per_class, len(idx))
        chosen.extend(rng.choice(idx, size=take, replace=False).tolist())
    remaining = [i for i in range(len(y)) if i not in chosen]
    while len(chosen) < min(max_rows, len(y)):
        chosen.append(int(rng.choice(remaining)))
        remaining = [i for i in remaining if i not in chosen]
    return np.array(chosen, dtype=int)


def value_rows_train(X, y):
    x_train, x_test, y_train, y_test = stable_split(X, y)
    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train)
    x_test_scaled = scaler.transform(x_test)
    val_size = min(60, len(y_test))
    X_val = x_test_scaled[:val_size]
    y_val = y_test[:val_size]
    sample_idx = pick_small_stratified(y_train, max_rows=8)
    phi, _ = exact_shapley(x_train_scaled[sample_idx], y_train[sample_idx], X_val, y_val)
    drop_global = sample_idx[phi < -0.01]
    keep_mask = np.ones(len(y_train), dtype=bool)
    keep_mask[drop_global] = False
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_train_scaled[keep_mask], y_train[keep_mask])
    pred = clf.predict(x_test_scaled)
    accuracy = accuracy_score(y_test, pred)
    return {
        "accuracy": float(accuracy),
        "sample_idx": sample_idx,
        "values": phi,
        "dropped": drop_global,
        "artifact": x_train_scaled[sample_idx, :2],
    }


## The dataset ladder

In [ ]:

rungs = dataquality_ladder()
preview_ladder(rungs)


## Run the same method across D1-D5

In [ ]:

results = []
for name, X, y in rungs:
    result = value_rows_train(X, y)
    row = {
        "rung": name,
        "accuracy": result["accuracy"],
        "min_value": float(result["values"].min()),
        "max_value": float(result["values"].max()),
        "dropped": int(len(result["dropped"])),
        "artifact": result["artifact"],
        "values": result["values"],
    }
    results.append(row)
    print(f"{name:28s} acc={row['accuracy']:.3f} min_phi={row['min_value']:.3f} max_phi={row['max_value']:.3f} dropped={row['dropped']}")


## Results visualization

In [ ]:

fig, axes = plt.subplots(1, 5, figsize=(16, 3))
for ax, row in zip(axes, results):
    art = row["artifact"]
    vals = row["values"]
    sc = ax.scatter(art[:, 0], art[:, 1], c=vals, cmap="coolwarm", s=30)
    ax.set_title(row["rung"].split()[0])
    ax.set_xlabel("valued f0")
    ax.set_ylabel("valued f1")
fig.suptitle("Small exact Shapley row values")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 3))
plt.plot(range(1, 6), [r["accuracy"] for r in results], marker="o")
plt.xticks(range(1, 6), ["D1", "D2", "D3", "D4", "D5"])
plt.ylim(0.6, 1.0)
plt.ylabel("accuracy after valuation action")
plt.title("Valuation-guided training by data quality")
plt.grid(True)
plt.show()


## Pitfall on D5: exact values at large scale

Exact Shapley sums over $2^n$ subsets, so a real messy dataset needs seeded approximation, convergence checks, and stable snapshots.

In [ ]:

def approximate_shapley_permutations(X_train, y_train, X_val, y_val, permutations=40):
    rng = np.random.default_rng(1018)
    n = len(y_train)
    values = np.zeros(n)
    for _ in range(permutations):
        order = rng.permutation(n)
        current = tuple()
        current_utility = utility_1nn(current, X_train, y_train, X_val, y_val)
        for idx in order:
            next_subset = tuple(sorted(current + (int(idx),)))
            next_utility = utility_1nn(next_subset, X_train, y_train, X_val, y_val)
            values[idx] += next_utility - current_utility
            current = next_subset
            current_utility = next_utility
    return values / permutations


name, X, y = rungs[-1]
x_train, x_test, y_train, y_test = stable_split(X, y)
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)
small_idx = pick_small_stratified(y_train, max_rows=12)
subset_count = 2 ** len(small_idx)
approx_one = approximate_shapley_permutations(x_train_scaled[small_idx], y_train[small_idx], x_test_scaled[:50], y_test[:50], permutations=30)
approx_two = approximate_shapley_permutations(x_train_scaled[small_idx], y_train[small_idx], x_test_scaled[:50], y_test[:50], permutations=60)
mean_error = float(np.mean(np.abs(approx_one - approx_two)))
print("exact subsets for 12 rows", subset_count)
print("mean approximation difference", round(mean_error, 4))
assert subset_count == 4096
assert mean_error < 0.20



## Evaluate it + Practice

- Compare the reported accuracy to a no-skill majority-class baseline for every rung.
- Sanity-check that the key data artifact changes in the intended direction before trusting the metric.
- Ablate the key data-centric step, such as filtering, augmentation, validation, version selection, or valuation-guided dropping.
- Watch for failure signals: gains only on corrupted validation data, impossible feature values, leakage, or unstable row rankings.
- Re-run with a new seed and require the conclusion, not every decimal, to remain stable.

Practice 1: change the corruption or repair strength and re-run the D1 to D5 curve.


Practice 2: add one slice-level diagnostic for the hardest rung.

Practice 3: replace accuracy with balanced accuracy or F1 and compare the story.